# 06 — Problem 1A: Lead-Time Regression at Checkout

**Question.** How many days from purchase to the customer's door?
**Stakeholder.** Operations — a number to quote, and an input to the promise.
**Metric.** MAE. Phase 1 showed lead time falls 5.34 days between the training and
test periods; under a level shift that size, R² against a variance computed on the
test period goes negative even for a useful model, so R² is a drift diagnostic
here rather than a score.

Two reference points frame everything: a constant fitted on the training period,
and **Olist's own delivery promise** — the incumbent any deployed model has to
beat.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config, evaluate as E, modeling as M
from data_load import load_all, load_analysis
from features import *
from viz import use_report_style, save_fig

use_report_style()
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "report" / "results"
def results(name):
    """Read one canonical result table written by src/train_models.py."""
    p = RESULTS / name
    assert p.exists(), f"{p} missing - run `python src/train_models.py` first"
    return pd.read_csv(p)
print("seed", config.SEED, "| window", config.ANALYSIS_START.date(), "->", config.ANALYSIS_END.date())

seed 42 | window 2017-01-01 -> 2018-08-31


In [2]:
reg = results("reg_p1a.csv")
show = reg[["model", "family", "rung", "cv_mae", "cv_std", "val_mae", "mae", "rmse", "r2"]]
display(show.round(3))

,model,family,rung,cv_mae,cv_std,val_mae,mae,rmse,r2
0,Baseline: training-period mean,Baseline,-1,NaN,NaN,NaN,6.645,7.572,-0.990
1,Baseline: Olist's own delivery promise,Baseline,-1,NaN,NaN,NaN,9.788,12.155,-4.130
2,LinearRegression,Linear,0,5.242,1.003,4.965,4.181,5.639,-0.104
3,Ridge,Linear,1,5.201,0.950,5.164,4.309,5.775,-0.158
4,DecisionTreeRegressor,Tree-based,0,5.539,0.906,5.927,6.116,8.664,-1.606
5,RandomForestRegressor,Tree-based,1,5.457,0.902,5.513,4.956,6.169,-0.321
6,HistGradientBoostingRegressor,Tree-based,2,5.495,0.813,4.802,4.141,5.517,-0.057
7,VotingRegressor (A+B),Ensemble,0,5.315,0.866,4.784,3.898,5.342,0.009
8,StackingRegressor (A+B),Ensemble,1,5.443,0.841,4.792,4.121,5.430,-0.024


## 6.1 The two baselines

Note the second row. The platform's own promised date is a *worse* point forecast
than a constant. That is expected rather than anomalous: a promised date is a
commercial decision, padded because a broken promise costs returns and retention
(Cui et al., 2024). It is a useful feature and a poor prediction.

In [3]:
base = reg[reg["family"] == "Baseline"]
for _, r in base.iterrows():
    print(f"  {r['model']:44s} MAE {r['mae']:5.2f}   RMSE {r['rmse']:5.2f}   R2 {r['r2']:+.3f}")

best = reg[reg["family"] != "Baseline"].sort_values("mae").iloc[0]
mean_row = base[base["model"].str.contains("mean")].iloc[0]
promise = base[base["model"].str.contains("promise")].iloc[0]
print(f"\nbest model: {best['model']}  MAE {best['mae']:.2f}")
print(f"  vs the constant : {100*(1-best['mae']/mean_row['mae']):.1f}% lower error")
print(f"  vs the promise  : {100*(1-best['mae']/promise['mae']):.1f}% lower error")

  Baseline: training-period mean               MAE  6.65   RMSE  7.57   R2 -0.990
  Baseline: Olist's own delivery promise       MAE  9.79   RMSE 12.16   R2 -4.130

best model: VotingRegressor (A+B)  MAE 3.90
  vs the constant : 41.3% lower error
  vs the promise  : 60.2% lower error


## 6.2 The complexity ladders

Each family starts at its simplest variant, and every rung above has to beat the
one below. `rung 0` is the family baseline the brief asks for.

In [4]:
for fam in ("Linear", "Tree-based", "Ensemble"):
    part = reg[reg["family"] == fam].sort_values("rung")
    if part.empty:
        continue
    print(f"\n{fam}")
    prev = None
    for _, r in part.iterrows():
        delta = "" if prev is None else f"   {r['val_mae'] - prev:+.3f} vs the rung below"
        print(f"  rung {int(r['rung'])}  {r['model']:36s} cvMAE {r['cv_mae']:6.3f}"
              f"   valMAE {r['val_mae']:5.3f}   testMAE {r['mae']:5.3f}{delta}")
        prev = r["val_mae"]


Linear
  rung 0  LinearRegression                     cvMAE  5.242   valMAE 4.965   testMAE 4.181
  rung 1  Ridge                                cvMAE  5.201   valMAE 5.164   testMAE 4.309   +0.199 vs the rung below

Tree-based
  rung 0  DecisionTreeRegressor                cvMAE  5.539   valMAE 5.927   testMAE 6.116
  rung 1  RandomForestRegressor                cvMAE  5.457   valMAE 5.513   testMAE 4.956   -0.414 vs the rung below
  rung 2  HistGradientBoostingRegressor        cvMAE  5.495   valMAE 4.802   testMAE 4.141   -0.711 vs the rung below

Ensemble
  rung 0  VotingRegressor (A+B)                cvMAE  5.315   valMAE 4.784   testMAE 3.898
  rung 1  StackingRegressor (A+B)              cvMAE  5.443   valMAE 4.792   testMAE 4.121   +0.008 vs the rung below


## 6.3 The Phase 1 re-check: is plain least squares really unusable?

The Phase 1 report stated that ordinary least squares was unusable here — *"the
design matrix is rank-deficient, so its coefficients explode"*. Phase 2 re-ran it
four ways. The observation reproduces; the conclusion drawn from it does not.

In [5]:
enc = results("encoding_diagnostic.csv")
display(enc.assign(
    max_abs_coefficient=lambda d: d["max_abs_coefficient"].map(lambda v: f"{v:.2e}"),
    cv_mae=lambda d: d["cv_mae"].map(lambda v: f"{v:,.3f}"))
    [["encoding", "rank", "encoded_columns", "deficiency",
      "max_abs_coefficient", "cv_mae", "test_mae"]])

,encoding,rank,encoded_columns,deficiency,max_abs_coefficient,cv_mae,test_mae
0,"LinearRegression, full one-hot block",101,105,4,2.72e+13,"392,238,155,930.966",4.181292
1,"LinearRegression, one level dropped",101,101,0,1.47e+01,5.242,4.181289
2,"Ridge (alpha=1), full one-hot block",101,105,4,9.82e+00,5.241,4.181818
3,"Ridge (alpha=1), one level dropped",101,101,0,1.25e+01,5.240,4.183486


Read across the rows:

* The deficiency is **exactly four** — one per categorical, because each one-hot
  block sums to one and is therefore collinear with the intercept. Phase 1's
  diagnosis was right.
* Least squares still returns a minimum-norm solution, and its *predictions* are
  fine — the test MAE is identical to Ridge's to three decimals. What is not fine
  is the coefficient vector, at around 10¹³.
* That cancellation is fragile. It holds on four of five expanding-window folds
  and fails on the fifth, which is why the CV error is astronomical while the
  single-split test error looks healthy. **A single train/test split would never
  have shown this.**
* Dropping one level per categorical restores full rank and sane coefficients.
  Phase 2 therefore encodes linear models with `drop="first"`.

The last row carries the real lesson. Once the design is full rank, Ridge and OLS
agree to three decimals. Regularisation buys numerical safety here, not accuracy —
accuracy comes from dropping linearity altogether.

## 6.4 How the winner is chosen

Not on the test column, and not on any single window. Every candidate is scored
on six out-of-sample windows — the five expanding CV folds and the validation
period — ranked within each, and judged on its mean rank over the **three most
recent**.

Two failure modes forced that rule, and both are visible in the table above.

* **Averaging raw scores across folds fails.** The 2017 folds are much harder
  (12–17 day lead times against 8.3 in the test window), so cross-validated MAE
  runs about a day high for everything and the *ordering* does not survive: CV
  puts Ridge above the boosted trees, every later window puts them the other way
  round, and so does the test period.
* **Using validation alone fails too** — it is two months, and notebook 07 shows
  what that costs on the late-delivery problem.

Ranks are used rather than scores because ranks are the only thing comparable
across windows of different difficulty. All six are reported; the recent three
vote.

In [6]:
sel = results("selection_p1a.csv")
cols = ["model"] + [c for c in sel.columns if c.startswith("rank_")] +        ["mean_rank_all", "mean_rank"]
display(sel[cols].round(2))
print(f"windows voting: {sel['windows_voting'].iloc[0]}")
print("\nNote the disagreement between mean_rank_all (all six windows) and "
      "mean_rank (the recent three).")

,model,rank_fold1,rank_fold2,rank_fold3,rank_fold4,rank_fold5,rank_validation,mean_rank_all,mean_rank
0,VotingRegressor (A+B),5.0,1.0,1.0,3.0,3.0,1.0,2.33,2.33
1,StackingRegressor (A+B),6.0,2.0,3.0,1.0,5.0,2.0,3.17,2.67
2,Ridge,1.0,5.0,2.0,4.0,2.0,5.0,3.17,3.67
3,HistGradientBoostingRegressor,7.0,4.0,4.0,5.0,4.0,3.0,4.50,4.00
4,LinearRegression,2.0,3.0,6.0,7.0,1.0,4.0,3.83,4.00
5,RandomForestRegressor,3.0,6.0,5.0,2.0,7.0,6.0,4.83,5.00
6,DecisionTreeRegressor,4.0,7.0,7.0,6.0,6.0,7.0,6.17,6.33


windows voting: fold4, fold5, validation

Note the disagreement between mean_rank_all (all six windows) and mean_rank (the recent three).


In [7]:
import json
meta = json.loads((config.MODELS_DIR / "model_metadata.json").read_text("utf-8"))
p1a = meta["p1a"]
print(f"selected  : {p1a['label']}  ({p1a['family']} family)")
print(f"validation: MAE {p1a['val_mae']:.3f}   cross-validated: {p1a['cv_mae']:.3f}")
print(f"test      : MAE {p1a['test_mae']:.3f}   RMSE {p1a['test_rmse']:.3f}   "
      f"R2 {p1a['test_r2']:+.3f}")
print(f"params    : {p1a['params']}")

reg = results("reg_p1a.csv")
mean_row = reg[reg["model"].str.contains("training-period mean")].iloc[0]
promise = reg[reg["model"].str.contains("promise")].iloc[0]
print(f"\nagainst a constant fitted on the past : "
      f"{100*(1-p1a['test_mae']/mean_row['mae']):.1f}% lower error")
print(f"against Olist's own promised date    : "
      f"{100*(1-p1a['test_mae']/promise['mae']):.1f}% lower error")

import figures_phase2 as F
print("\n" + Path(F.fig_regression_ladder()).name)
print(Path(F.fig_regression_residuals()).name)
print(Path(F.fig_extrapolation()).name)

selected  : VotingRegressor (A+B)  (Ensemble family)
validation: MAE 4.784   cross-validated: 5.315
test      : MAE 3.898   RMSE 5.342   R2 +0.009
params    : {}

against a constant fitted on the past : 41.3% lower error
against Olist's own promised date    : 60.2% lower error



fig10_regression_ladder.pdf


fig11_regression_residuals.pdf


fig19_extrapolation.pdf


In [8]:
seg = results("segments_p1a_month.csv")
display(seg.round(3))
assert seg["mae"].max() < 3 * seg["mae"].min(), "error varies wildly across test months"
print("\nBias is the number to watch: a positive value means the model, fitted on a "
      "slower past, over-predicts how long the parcel will take.")

,_month,n,mae,bias,actual_mean
0,2018-08,6351,3.374,1.203,7.733
1,2018-07,6156,4.417,2.261,8.959



Bias is the number to watch: a positive value means the model, fitted on a slower past, over-predicts how long the parcel will take.
